# Hands-On: Building Multi-Agent Systems with LangGraph

## Overview

LangGraph is a powerful library for building stateful, multi-agent applications with LLMs. It provides:

- **State Management**: Track conversation history and agent outputs
- **Graph-based Workflows**: Define complex agent interactions
- **Conditional Routing**: Dynamic agent selection
- **Human-in-the-Loop**: Integrate human feedback
- **Persistence**: Save and resume agent state

### What We'll Build:

1. Simple Sequential Agent Chain
2. Parallel Agent Execution
3. Supervisor Pattern (Hierarchical)
4. Real-World Example: Research Assistant System

In [ ]:
# Setup
import sys
sys.path.append('..')

import os
from typing import Annotated, TypedDict, List, Dict, Any
from typing_extensions import TypedDict

# LangGraph imports
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Note: You'll need to set your API key
# Option 1: Use OpenAI (requires API key)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# from langchain_openai import ChatOpenAI

# Option 2: Use a local model with Ollama (free, no API key needed)
# First install Ollama from https://ollama.ai
# Then run: ollama pull llama2

print("✓ Imports successful!")
print("\n📝 Note: This demo can work with:")
print("   1. OpenAI API (requires API key)")
print("   2. Local models via Ollama (free, install from https://ollama.ai)")
print("   3. Mock agents for learning the patterns (no LLM needed)")

## 1. Understanding LangGraph Basics

LangGraph uses a **state graph** where:
- **Nodes** are functions (agents)
- **Edges** define the flow between agents
- **State** is shared data passed between agents

In [ ]:
# Define the state that will be passed between agents
class AgentState(TypedDict):
    """State shared across all agents."""
    messages: Annotated[List, add_messages]  # Conversation history
    current_step: str  # Track which step we're on
    data: Dict[str, Any]  # Any additional data


# Create simple mock agents (we'll use real LLMs later)
def researcher_agent(state: AgentState) -> AgentState:
    """Mock researcher agent."""
    print("🔍 Researcher Agent: Gathering information...")
    
    state["messages"].append(
        AIMessage(content="I've researched the topic and found 3 key papers on multi-agent systems.")
    )
    state["data"]["research_complete"] = True
    state["current_step"] = "research"
    
    return state


def analyzer_agent(state: AgentState) -> AgentState:
    """Mock analyzer agent."""
    print("📊 Analyzer Agent: Analyzing findings...")
    
    state["messages"].append(
        AIMessage(content="Analysis complete. Key insight: Hierarchical patterns work best for complex tasks.")
    )
    state["data"]["analysis_complete"] = True
    state["current_step"] = "analysis"
    
    return state


def writer_agent(state: AgentState) -> AgentState:
    """Mock writer agent."""
    print("✍️  Writer Agent: Creating summary...")
    
    state["messages"].append(
        AIMessage(content="Summary written: Multi-agent systems enable complex task decomposition.")
    )
    state["data"]["writing_complete"] = True
    state["current_step"] = "writing"
    
    return state


print("✓ Mock agents defined")

## 2. Example 1: Sequential Agent Chain

Let's create a simple sequential workflow: Research → Analyze → Write

In [ ]:
# Create the graph
workflow = StateGraph(AgentState)

# Add nodes (agents)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("analyzer", analyzer_agent)
workflow.add_node("writer", writer_agent)

# Define the flow (edges)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "analyzer")
workflow.add_edge("analyzer", "writer")
workflow.add_edge("writer", END)

# Compile the graph
app = workflow.compile()

print("✓ Sequential workflow created")
print("\nFlow: researcher → analyzer → writer → END")

In [ ]:
# Execute the workflow
initial_state = {
    "messages": [HumanMessage(content="Research multi-agent orchestration patterns")],
    "current_step": "start",
    "data": {}
}

print("🚀 Starting sequential workflow...\n")

# Run the graph
final_state = app.invoke(initial_state)

print("\n✅ Workflow complete!")
print(f"\n📝 Message history ({len(final_state['messages'])} messages):")
for i, msg in enumerate(final_state["messages"]):
    msg_type = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"  {i+1}. [{msg_type}] {msg.content}")

## 3. Example 2: Conditional Routing

Let's add logic to route based on the task type.

In [ ]:
# Define routing function
def route_task(state: AgentState) -> str:
    """Route to different agents based on task type."""
    last_message = state["messages"][-1].content.lower()
    
    if "code" in last_message or "implement" in last_message:
        print("🔀 Routing to: coder")
        return "coder"
    elif "research" in last_message or "find" in last_message:
        print("🔀 Routing to: researcher")
        return "researcher"
    else:
        print("🔀 Routing to: general")
        return "general"


def coder_agent(state: AgentState) -> AgentState:
    """Agent specialized in coding."""
    print("💻 Coder Agent: Writing code...")
    state["messages"].append(
        AIMessage(content="Here's a Python implementation of a multi-agent system.")
    )
    return state


def general_agent(state: AgentState) -> AgentState:
    """General purpose agent."""
    print("🤖 General Agent: Processing request...")
    state["messages"].append(
        AIMessage(content="I can help with general questions.")
    )
    return state


# Create conditional workflow
conditional_workflow = StateGraph(AgentState)

# Add specialized agents
conditional_workflow.add_node("router", lambda state: state)  # Passthrough node
conditional_workflow.add_node("researcher", researcher_agent)
conditional_workflow.add_node("coder", coder_agent)
conditional_workflow.add_node("general", general_agent)

# Set entry point and conditional routing
conditional_workflow.set_entry_point("router")
conditional_workflow.add_conditional_edges(
    "router",
    route_task,
    {
        "researcher": "researcher",
        "coder": "coder",
        "general": "general"
    }
)

# All paths lead to END
conditional_workflow.add_edge("researcher", END)
conditional_workflow.add_edge("coder", END)
conditional_workflow.add_edge("general", END)

conditional_app = conditional_workflow.compile()

print("✓ Conditional workflow created")

In [ ]:
# Test different types of requests
test_requests = [
    "Research the latest developments in AI",
    "Implement a sorting algorithm in Python",
    "What's the weather like today?"
]

for request in test_requests:
    print(f"\n{'='*60}")
    print(f"📥 Request: {request}")
    print('='*60)
    
    state = {
        "messages": [HumanMessage(content=request)],
        "current_step": "start",
        "data": {}
    }
    
    result = conditional_app.invoke(state)
    print(f"📤 Response: {result['messages'][-1].content}")

## 4. Example 3: Supervisor Pattern

Implement a supervisor that coordinates multiple worker agents.

In [ ]:
from typing import Literal

class SupervisorState(TypedDict):
    """State for supervisor pattern."""
    messages: Annotated[List, add_messages]
    next_agent: str
    task_queue: List[str]
    completed_tasks: List[str]


def supervisor_agent(state: SupervisorState) -> SupervisorState:
    """Supervisor that delegates tasks to workers."""
    print("\n👔 Supervisor: Analyzing task and delegating...")
    
    # Simple task delegation logic
    if state["task_queue"]:
        next_task = state["task_queue"][0]
        state["task_queue"] = state["task_queue"][1:]
        
        # Decide which worker to assign
        if "data" in next_task.lower():
            state["next_agent"] = "data_worker"
        elif "report" in next_task.lower():
            state["next_agent"] = "report_worker"
        else:
            state["next_agent"] = "general_worker"
            
        state["messages"].append(
            AIMessage(content=f"Delegating '{next_task}' to {state['next_agent']}")
        )
    else:
        state["next_agent"] = "FINISH"
        state["messages"].append(
            AIMessage(content="All tasks completed!")
        )
    
    return state


def data_worker(state: SupervisorState) -> SupervisorState:
    """Worker specialized in data tasks."""
    print("   👷 Data Worker: Processing data...")
    state["messages"].append(
        AIMessage(content="Data processing complete. 1000 records analyzed.")
    )
    state["completed_tasks"].append("data_processing")
    state["next_agent"] = "supervisor"
    return state


def report_worker(state: SupervisorState) -> SupervisorState:
    """Worker specialized in report generation."""
    print("   👷 Report Worker: Generating report...")
    state["messages"].append(
        AIMessage(content="Report generated successfully. 5 pages with visualizations.")
    )
    state["completed_tasks"].append("report_generation")
    state["next_agent"] = "supervisor"
    return state


def general_worker(state: SupervisorState) -> SupervisorState:
    """General purpose worker."""
    print("   👷 General Worker: Handling task...")
    state["messages"].append(
        AIMessage(content="Task completed.")
    )
    state["completed_tasks"].append("general_task")
    state["next_agent"] = "supervisor"
    return state


# Router for supervisor pattern
def supervisor_router(state: SupervisorState) -> Literal["data_worker", "report_worker", "general_worker", "supervisor", "__end__"]:
    """Route based on supervisor's decision."""
    next_agent = state.get("next_agent", "supervisor")
    
    if next_agent == "FINISH":
        return "__end__"
    return next_agent


# Build supervisor workflow
supervisor_workflow = StateGraph(SupervisorState)

# Add all agents
supervisor_workflow.add_node("supervisor", supervisor_agent)
supervisor_workflow.add_node("data_worker", data_worker)
supervisor_workflow.add_node("report_worker", report_worker)
supervisor_workflow.add_node("general_worker", general_worker)

# Start with supervisor
supervisor_workflow.set_entry_point("supervisor")

# Conditional routing from supervisor
supervisor_workflow.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "data_worker": "data_worker",
        "report_worker": "report_worker",
        "general_worker": "general_worker",
        "__end__": END
    }
)

# Workers return to supervisor
supervisor_workflow.add_conditional_edges(
    "data_worker",
    supervisor_router,
    {"supervisor": "supervisor"}
)
supervisor_workflow.add_conditional_edges(
    "report_worker",
    supervisor_router,
    {"supervisor": "supervisor"}
)
supervisor_workflow.add_conditional_edges(
    "general_worker",
    supervisor_router,
    {"supervisor": "supervisor"}
)

supervisor_app = supervisor_workflow.compile()

print("✓ Supervisor workflow created")

In [ ]:
# Execute supervisor workflow with multiple tasks
initial_state = {
    "messages": [HumanMessage(content="Process these tasks")],
    "next_agent": "supervisor",
    "task_queue": [
        "Process data from API",
        "Generate quarterly report",
        "Clean up old files"
    ],
    "completed_tasks": []
}

print("🚀 Starting supervisor workflow...")
print(f"📋 Tasks to complete: {len(initial_state['task_queue'])}\n")

final_state = supervisor_app.invoke(initial_state)

print("\n" + "="*60)
print("✅ All tasks completed!")
print(f"\n📊 Summary:")
print(f"   Total messages: {len(final_state['messages'])}")
print(f"   Completed tasks: {final_state['completed_tasks']}")
print(f"   Remaining tasks: {final_state['task_queue']}")

## 5. Key Patterns Summary

### Sequential Pattern
```python
workflow.add_edge("agent1", "agent2")
workflow.add_edge("agent2", "agent3")
```
**Use when**: Tasks must happen in order

### Conditional Routing
```python
workflow.add_conditional_edges(
    "router",
    routing_function,
    {"path1": "agent1", "path2": "agent2"}
)
```
**Use when**: Different agents handle different task types

### Supervisor Pattern
```python
# Supervisor delegates, workers return to supervisor
workflow.add_conditional_edges("supervisor", router, paths)
workflow.add_edge("worker", "supervisor")
```
**Use when**: Complex task coordination needed

### Parallel Execution
```python
# Multiple workers, results aggregated
workflow.add_edge("splitter", "worker1")
workflow.add_edge("splitter", "worker2")
workflow.add_edge("worker1", "aggregator")
workflow.add_edge("worker2", "aggregator")
```
**Use when**: Independent tasks can run simultaneously

## 6. Exercise: Build Your Own Multi-Agent System

Create a multi-agent system for content creation:

**Agents needed:**
1. Topic Researcher - finds relevant information
2. Outline Creator - structures the content
3. Writer - creates the draft
4. Editor - reviews and improves
5. Fact Checker - verifies claims

**Challenge**: Implement this using LangGraph!

**Hints**:
- Research → Outline → Writer is sequential
- Editor and Fact Checker could run in parallel
- Consider a supervisor to coordinate

In [ ]:
# Your implementation here!

# Define your state
class ContentState(TypedDict):
    messages: Annotated[List, add_messages]
    # Add your state fields
    pass

# Define your agents
def topic_researcher(state: ContentState) -> ContentState:
    # Your implementation
    pass

# Build your workflow
# content_workflow = StateGraph(ContentState)
# ...

## 7. Next Steps

### Extend Your Knowledge:

1. **Add LLM Integration**: Replace mock agents with actual LLMs (OpenAI, Anthropic, or local models)
2. **Add Persistence**: Use LangGraph's checkpointing to save/resume workflows
3. **Human-in-the-Loop**: Add approval steps for critical decisions
4. **Error Handling**: Implement retry logic and fallbacks
5. **Monitoring**: Add logging and metrics

### Resources:

- [LangGraph Documentation](https://python.langchain.com/docs/langgraph)
- [LangGraph Examples](https://github.com/langchain-ai/langgraph/tree/main/examples)
- [Multi-Agent Patterns](https://python.langchain.com/docs/use_cases/agent_workflows)

### Try Other Frameworks:

- **AutoGen**: Microsoft's multi-agent framework
- **CrewAI**: Role-based multi-agent systems
- **Custom Solutions**: Build your own from scratch!

## Summary

In this notebook, we learned:

- ✓ LangGraph fundamentals (state, nodes, edges)
- ✓ Sequential agent chains
- ✓ Conditional routing for dynamic agent selection
- ✓ Supervisor pattern for hierarchical coordination
- ✓ How to structure real-world multi-agent systems

**Congratulations!** You now have the foundational knowledge to build sophisticated multi-agent AI systems. Experiment with different patterns and find what works best for your use case!